In [1]:
from glob import glob

files = glob('chunk-30s-dialects/*.mp3')
len(files)

879595

In [2]:
import requests

In [3]:
%%time

request = {
    'file': open(files[0], 'rb'),
    'model': (None, 'openai/whisper-large-v3'),
    'response_format': (None, 'json'),
    'language': (None, 'ms')
}

response = requests.post('http://localhost:8000/v1/audio/transcriptions', files=request)
response.json()

CPU times: user 12 ms, sys: 65 µs, total: 12 ms
Wall time: 3.4 s


{'text': ' Bagi yang nak beri sumbangan melalui online. Surah Mahad Baitur Rahman Banggu Donah kita menggunakan akaun bank muamalat 1301 0006 374717. Akaun TMB kalau nak bayar biletrik 2103 2304 0303. Atas nama Muhammad Muzani bin Abdul Rahman. Pelan-pelan kutip. Semua orang akan bagi sumbangan. Sedikit atau banyak ikhlas.'}

In [4]:
import IPython.display as ipd
ipd.Audio(files[0])

In [5]:
folder = 'chunk-30s-dialects-whisper'
!mkdir {folder}

mkdir: cannot create directory ‘chunk-30s-dialects-whisper’: File exists


In [6]:
import os
import json

def generate_answer(row):

    f = os.path.split(row)[1]
    filename = os.path.join(folder, f)
    try:
        with open(filename) as fopen:
            json.load(fopen)
        return
    except:
        pass
    
    for _ in range(3):
        try:

            request = {
                'file': open(row, 'rb'),
                'model': (None, 'openai/whisper-large-v3'),
                'response_format': (None, 'json'),
                'language': (None, 'ms')
            }
            
            response = requests.post('http://localhost:8000/v1/audio/transcriptions', files=request)
            r = response.json()
            with open(filename, 'w') as fopen:
                json.dump(r['text'].strip(), fopen)
                return
        except Exception as e:
            pass

In [7]:
generate_answer(files[0])

In [8]:
def consumer(queue, name):
    while True:
        if queue.qsize() == 0:
            break
        item = queue.get()
        generate_answer(item)
    print(f'consumer {name} done')

In [9]:
from threading import Thread
from queue import Queue

queue = Queue()
for u in files:
    queue.put(u)
    
ori_size = queue.qsize()

In [10]:
from tqdm import tqdm

max_worker = 100
consumers = [Thread(target=consumer, args=(queue,i)) for i in range(max_worker)]
for i in range(len(consumers)):
    consumers[i].start()
    
pbar = tqdm(total=ori_size)
last_size = 0
while True:
    size = queue.qsize()
    if size == 0:
        break
    left = ori_size - size
    minus = left - last_size
    if minus > 0:
        pbar.update(minus)
        last_size += minus

pbar.close()

100%|█████████▉| 879593/879595 [01:13<00:00, 12036.91it/s]

consumer 81 done
consumer 91 doneconsumer 72 done

consumer 6 done
consumer 63 done
consumer 1 done
consumer 97 done
consumer 19 done
consumer 2 done
consumer 53 done
consumer 28 done
consumer 84 done
consumer 59 done
consumer 9 done
consumer 82 done
consumer 61 done
consumer 48 done
consumer 52 done
consumer 37 done
consumer 25 done
consumer 11 done
consumer 7 done
consumer 34 done
consumer 21 done
consumer 43 done
consumer 44 done
consumer 67 done
consumer 88 done
consumer 49 doneconsumer 68 done

consumer 35 done
consumer 66 done
consumer 99 done
consumer 18 done
consumer 70 done
consumer 23 done
consumer 87 doneconsumer 56 done

consumer 94 done
consumer 96 doneconsumer 64 done
consumer 47 done
consumer 69 done

consumer 83 done
consumer 79 done
consumer 17 done
consumer 39 done
consumer 54 done
consumer 5 done
consumer 80 done
consumer 32 done
consumer 71 done
consumer 45 done
consumer 20 done
consumer 46 done
consumer 65 done
consumer 90 done
consumer 62 done
consumer 40 done
con

consumer 36 done
consumer 76 done
consumer 16 done
consumer 85 done
consumer 3 done
consumer 8 done
consumer 29 done
consumer 55 done
consumer 86 done
consumer 4 doneconsumer 13 done



In [11]:
files = glob('chunk-30s-dialects-whisper/*')
len(files)

879595

In [16]:
transcription = []
for f in tqdm(files):
    with open(f) as fopen:
        d = json.load(fopen)
    transcription.append({
        'audio_filename': f.replace('-whisper/', '/'),
        'text': d,
    })

100%|██████████| 879595/879595 [00:10<00:00, 86025.60it/s]


In [18]:
from datasets import Dataset

dataset = Dataset.from_list(transcription)

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
dataset.push_to_hub('malaysia-ai/pseudolabel-filtered-dialects-youtube-whisper-large-v3')

Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.83s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/pseudolabel-filtered-dialects-youtube-whisper-large-v3/commit/5fa808155496f425905945732456316aea475e78', commit_message='Upload dataset', commit_description='', oid='5fa808155496f425905945732456316aea475e78', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/pseudolabel-filtered-dialects-youtube-whisper-large-v3', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/pseudolabel-filtered-dialects-youtube-whisper-large-v3'), pr_revision=None, pr_num=None)